# Análisis Estadístico 


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path

# ── Rutas (relativas a la ubicación del notebook) ──────────────────────────────
_HERE     = Path(".").resolve()              # jupyter/notebooks/
_ROOT     = _HERE.parent.parent              # raíz del proyecto
_EVAL     = _ROOT / "src" / "evaluacion"

RUTA_CSV    = _EVAL / "resultados" / "detalle_completo.csv"
RUTA_LEXICAS = _EVAL / "resultados" / "metricas_lexicas_por_turno.csv"
RUTA_OUT    = RUTA_CSV.parent

df = pd.read_csv(RUTA_CSV, encoding="utf-8-sig")

# ── Integrar semantic_overlap desde métricas léxicas ───────────────────────────
if RUTA_LEXICAS.exists():
    df_lex = pd.read_csv(RUTA_LEXICAS, encoding="utf-8-sig")
    cols_merge = ["condicion", "actor_id", "turno"]
    cols_lex = cols_merge + ["semantic_overlap", "vocabulary_overlap",
                              "conflict_vocab_overlap", "oral_markers_adoption"]
    cols_disponibles = [c for c in cols_lex if c in df_lex.columns]
    df_lex_sub = df_lex[cols_disponibles].copy()
    df_lex_sub = df_lex_sub.groupby(cols_merge, as_index=False).mean(numeric_only=True)
    df = df.merge(df_lex_sub, on=cols_merge, how="left")
    print(f"Métricas léxicas integradas: {[c for c in cols_disponibles if c not in cols_merge]}")
else:
    print("⚠ No se encontró metricas_lexicas_por_turno.csv — semantic_overlap no disponible")

METRICAS = [
    "estabilidad_rol", "autenticidad_lexica", "autenticidad_emocional",
    "goal_directedness", "tactical_consistency", "factual_grounding",
    "answer_relevance",
    "semantic_overlap", "vocabulary_overlap", "conflict_vocab_overlap",
    "oral_markers_adoption",
]
METRICAS = [m for m in METRICAS if m in df.columns]

CONDICIONES = ["baseline", "rag_only", "bdi_only", "bdi_rag"]
PARES = [
    ("baseline", "rag_only"), ("baseline", "bdi_only"), ("baseline", "bdi_rag"),
    ("rag_only", "bdi_only"), ("rag_only", "bdi_rag"), ("bdi_only", "bdi_rag"),
]

for col in METRICAS + ["persona_score", "bdi_score", "rag_score"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Registros cargados: {len(df)}")
print(f"Condiciones: {df['condicion'].unique().tolist()}")
print(f"Métricas en análisis: {METRICAS}")
df.head(3)

## 1. Kruskal-Wallis
¿Hay diferencia estadísticamente significativa entre las 4 condiciones para cada métrica?


In [ ]:
def sig_label(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

filas_kw = []
for metrica in METRICAS:
    grupos = [df[df["condicion"]==c][metrica].dropna().values for c in CONDICIONES]
    grupos_validos = [g for g in grupos if len(g) >= 2]
    if len(grupos_validos) < 2:
        continue
    H, p = stats.kruskal(*grupos_validos)
    filas_kw.append({
        "metrica"       : metrica,
        "H"             : round(H, 3),
        "p_value"       : round(p, 4),
        "significancia" : sig_label(p),
        "conclusion"    : "diferencia significativa" if p < 0.05 else "sin diferencia significativa",
    })

df_kw = pd.DataFrame(filas_kw)
df_kw.to_csv(RUTA_OUT / "estadistica_kruskal_wallis.csv", index=False, encoding="utf-8-sig")
print("Leyenda: *** p<0.001 | ** p<0.01 | * p<0.05 | ns no significativo\n")
df_kw

## 2. Mann-Whitney U + Bonferroni + Cliff's Delta
Comparaciones par a par con corrección de múltiples comparaciones y tamaño de efecto.

In [ ]:
def cliff_delta(x, y):
    """Cliff's Delta — magnitud del efecto para datos ordinales. Rango [-1, 1]"""
    dom = sum(1 if xi > yi else -1 if xi < yi else 0 for xi in x for yi in y)
    return dom / (len(x) * len(y))

def magnitud_cliff(d):
    ad = abs(d)
    if ad >= 0.474: return "grande"
    if ad >= 0.330: return "mediano"
    if ad >= 0.147: return "pequeño"
    return "negligible"

n_pares = len(PARES)
filas_mw = []

for metrica in METRICAS:
    for c1, c2 in PARES:
        x = df[df["condicion"]==c1][metrica].dropna().values
        y = df[df["condicion"]==c2][metrica].dropna().values
        if len(x) < 2 or len(y) < 2:
            continue
        U, p_raw  = stats.mannwhitneyu(x, y, alternative="two-sided")
        p_bonf    = min(float(p_raw) * n_pares, 1.0)
        d         = cliff_delta(x, y)
        filas_mw.append({
            "metrica"      : metrica,
            "comparacion"  : f"{c1}  vs  {c2}",
            "U"            : round(U, 1),
            "p_raw"        : round(float(p_raw), 4),
            "p_bonferroni" : round(p_bonf, 4),
            "significancia": sig_label(p_bonf),
            "cliff_delta"  : round(d, 3),
            "magnitud"     : magnitud_cliff(d),
        })

df_mw = pd.DataFrame(filas_mw)
df_mw.to_csv(RUTA_OUT / "estadistica_mann_whitney.csv", index=False, encoding="utf-8-sig")
print("Cliff's delta: + significa que la primera condición supera a la segunda\n")
df_mw

## 3. Bootstrap IC 95%
Intervalo de confianza para cada media — cuánto podría variar si se repitiera el experimento.

In [ ]:
def bootstrap_ci(data, n_boot=2000, ci=0.95, seed=42):
    rng  = np.random.default_rng(seed)
    data = data[~np.isnan(data)]
    if len(data) == 0:
        return (float("nan"), float("nan"))
    boots = [rng.choice(data, size=len(data), replace=True).mean() for _ in range(n_boot)]
    alpha = (1 - ci) / 2
    return (float(np.percentile(boots, alpha*100)), float(np.percentile(boots, (1-alpha)*100)))

filas_ci = []
for metrica in METRICAS:
    for cond in CONDICIONES:
        datos = df[df["condicion"]==cond][metrica].dropna().values
        if len(datos) == 0:
            continue
        media        = float(np.mean(datos))
        ci_low, ci_hi = bootstrap_ci(datos)
        filas_ci.append({
            "metrica"   : metrica,
            "condicion" : cond,
            "n"         : len(datos),
            "media"     : round(media, 3),
            "IC_95_low" : round(ci_low, 3),
            "IC_95_high": round(ci_hi,  3),
            "IC_95"     : f"[{ci_low:.3f} – {ci_hi:.3f}]",
        })

df_ci = pd.DataFrame(filas_ci)
df_ci.to_csv(RUTA_OUT / "estadistica_bootstrap_ci.csv", index=False, encoding="utf-8-sig")
df_ci

## 4. Resumen visual — IC 95% por métrica
Intervalos de confianza de las 3 condiciones para comparar visualmente si se solapan.

In [ ]:
import matplotlib.pyplot as plt

colores = {"baseline": "#6c757d", "rag_only": "#0077b6", "bdi_only": "#e76f51", "bdi_rag": "#2d6a4f"}
offsets = {"baseline": -0.3, "rag_only": -0.1, "bdi_only": 0.1, "bdi_rag": 0.3}

# Separar métricas LLM (1-10) de métricas léxicas (0-1) para visualización
metricas_llm = [m for m in METRICAS if m not in
    ["semantic_overlap", "vocabulary_overlap", "conflict_vocab_overlap", "oral_markers_adoption"]]
metricas_lex = [m for m in METRICAS if m in
    ["semantic_overlap", "vocabulary_overlap", "conflict_vocab_overlap", "oral_markers_adoption"]]

fig, axes = plt.subplots(1, 2 if metricas_lex else 1,
                         figsize=(18 if metricas_lex else 14, 6),
                         gridspec_kw={"width_ratios": [2, 1]} if metricas_lex else None)

if not metricas_lex:
    axes = [axes]

# Panel 1: Métricas LLM (escala 1-10)
ax1 = axes[0]
for cond in CONDICIONES:
    subset = df_ci[(df_ci["condicion"] == cond) & (df_ci["metrica"].isin(metricas_llm))]
    if subset.empty:
        continue
    x_pos  = [metricas_llm.index(m) + offsets[cond] for m in subset["metrica"]]
    yerr   = [
        subset["media"].values - subset["IC_95_low"].values,
        subset["IC_95_high"].values - subset["media"].values,
    ]
    ax1.errorbar(
        x_pos, subset["media"].values, yerr=yerr,
        fmt="o", capsize=5, label=cond,
        color=colores[cond], linewidth=1.5, markersize=7,
    )

ax1.set_xticks(range(len(metricas_llm)))
ax1.set_xticklabels(metricas_llm, rotation=35, ha="right", fontsize=9)
ax1.set_ylabel("Score (1-10)", fontsize=11)
ax1.set_title("Métricas LLM-as-judge — IC 95% Bootstrap", fontsize=13)
ax1.legend(title="Condición", fontsize=9)
ax1.set_ylim(0.5, 10.5)
ax1.grid(axis="y", alpha=0.3)

# Panel 2: Métricas léxicas (escala 0-1)
if metricas_lex:
    ax2 = axes[1]
    for cond in CONDICIONES:
        subset = df_ci[(df_ci["condicion"] == cond) & (df_ci["metrica"].isin(metricas_lex))]
        if subset.empty:
            continue
        x_pos  = [metricas_lex.index(m) + offsets[cond] for m in subset["metrica"]]
        yerr   = [
            subset["media"].values - subset["IC_95_low"].values,
            subset["IC_95_high"].values - subset["media"].values,
        ]
        ax2.errorbar(
            x_pos, subset["media"].values, yerr=yerr,
            fmt="o", capsize=5, label=cond,
            color=colores[cond], linewidth=1.5, markersize=7,
        )
    ax2.set_xticks(range(len(metricas_lex)))
    ax2.set_xticklabels(metricas_lex, rotation=35, ha="right", fontsize=9)
    ax2.set_ylabel("Score (0-1)", fontsize=11)
    ax2.set_title("Métricas léxicas — IC 95% Bootstrap", fontsize=13)
    ax2.legend(title="Condición", fontsize=9)
    ax2.set_ylim(-0.05, 1.05)
    ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(RUTA_OUT / "bootstrap_ic_por_metrica.png", dpi=150)
plt.show()
print("Guardado: bootstrap_ic_por_metrica.png")

## Archivos generados

In [ ]:
archivos = [
    "estadistica_kruskal_wallis.csv",
    "estadistica_mann_whitney.csv",
    "estadistica_bootstrap_ci.csv",
    "bootstrap_ic_por_metrica.png",
]
for a in archivos:
    ruta = RUTA_OUT / a
    estado = "✓" if ruta.exists() else "✗ no encontrado"
    print(f"  {estado}  {a}")